In [13]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd


In [2]:
data = load_breast_cancer()
X = data.data
y = data.target


In [4]:
print("data shape", X.shape)
print("catogries", data.target_names)
print("Malignant (0):", (y == 0).sum(), "| Benign (1):", (y == 1).sum())

data shape (569, 30)
catogries ['malignant' 'benign']
Malignant (0): 212 | Benign (1): 357


## 2. Train-Test Split

I split the dataset into:
- **80% Training** — used to train the models
- **20% Testing** — used to evaluate the models

I use `stratify=y` to ensure both sets preserve the original class distribution,  
which is important since the dataset is slightly imbalanced.

In [5]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 455
Testing samples: 114



## 3. Model Training

I train three classification models using **default parameters**:

| Model | Note |
|-------|------|
| Logistic Regression | `max_iter=10000` to ensure convergence |
| SVM | Default parameters |
| KNN | Default parameters (k=5) |



In [26]:
# Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=10000)
lr_model.fit(X_train, y_train)



LogisticRegression(max_iter=10000, random_state=42)

In [8]:
# Support Vector Machine
svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)


SVC(random_state=42)

In [9]:
# K-Nearest Neighbors
knn_model = KNeighborsClassifier()
knn_model.fit(X_train, y_train)

print("All models trained successfully!")

All models trained successfully!



## 4. Model Evaluation

For each model, I compute the following metrics:

| Metric | Description |
|--------|-------------|
| **Accuracy** | Overall correct predictions |
| **Precision** | Of all predicted Benign, how many are actually Benign? |
| **Recall** | Of all actual Benign, how many did we correctly detect? |
| **F1-Score** | Balance between Precision and Recall |
| **Confusion Matrix** | Detailed breakdown of predictions |

> In a medical context, **Recall is the most critical metric** — missing a cancerous case is far more dangerous than a false alarm.

In [27]:
#Model Evaluation
models = {
    "Logistic Regression": lr_model,
    "SVM": svm_model,
    "KNN": knn_model
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    
    print(f"********** {name} **********")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
    print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
    print()

********** Logistic Regression **********
Accuracy:  0.9649
Precision: 0.9595
Recall:    0.9861
F1-Score:  0.9726
Confusion Matrix:
[[39  3]
 [ 1 71]]

********** SVM **********
Accuracy:  0.9298
Precision: 0.9211
Recall:    0.9722
F1-Score:  0.9459
Confusion Matrix:
[[36  6]
 [ 2 70]]

********** KNN **********
Accuracy:  0.9123
Precision: 0.9429
Recall:    0.9167
F1-Score:  0.9296
Confusion Matrix:
[[38  4]
 [ 6 66]]



### 5. Evaluation Results Summary

| Model | Accuracy | Precision | Recall | F1-Score |
|-------|----------|-----------|--------|----------|
| Logistic Regression | 0.9649 | 0.9595 | **0.9861** | **0.9726** |
| SVM | 0.9298 | 0.9211 | 0.9722 | 0.9459 |
| KNN | 0.9123 | 0.9429 | 0.9167 | 0.9296 |

#### Confusion Matrix Breakdown (Logistic Regression)
|  | Predicted Malignant | Predicted Benign |
|--|---------------------|-----------------|
| **Actual Malignant** | 39  | 3  |
| **Actual Benign** | 1  | 71  |

- **False Positives (3):** Predicted Benign but actually Malignant — dangerous in medical context
- **False Negatives (1):** Predicted Malignant but actually Benign



## 5. Model Comparison

In [14]:
# Model Comparison Table

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "F1-Score":  round(f1_score(y_test, y_pred), 4)
    })

df_results = pd.DataFrame(results)
df_results = df_results.sort_values("F1-Score", ascending=False).reset_index(drop=True)
print(df_results.to_string(index=False))

              Model  Accuracy  Precision  Recall  F1-Score
Logistic Regression    0.9649     0.9595  0.9861    0.9726
                SVM    0.9298     0.9211  0.9722    0.9459
                KNN    0.9123     0.9429  0.9167    0.9296




## Conclusion

### Which model performed best?
**Logistic Regression** achieved the best performance across all metrics:
- Highest Accuracy: **96.49%**
- Highest Recall: **98.61%**
- Highest F1-Score: **97.26%**

### Which metric matters most in a medical context?
**Recall** is the most important metric in cancer detection because:

> Missing a real cancer case (False Negative) is far more dangerous than  
> flagging a healthy patient for further tests (False Positive).

A model with high Recall ensures that **as few cancerous cases as possible go undetected**,  
which is critical for early treatment and patient survival.

**Logistic Regression** had the highest Recall (98.61%), making it the most suitable model  
for this medical classification task.


## Extra Section: Experimenting with Model Parameters

In this section, I test different parameter values for each model  
to see how they affect performance compared to the default settings.

In [30]:
print("Logistic Regression — Testing different max_iter")
print("*" * 60)
for max_iter in [100, 500, 1000, 5000, 10000]:
    model = LogisticRegression(random_state=42, max_iter=max_iter)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"max_iter = {max_iter:6} | Accuracy: {accuracy_score(y_test, y_pred):.4f} | Recall: {recall_score(y_test, y_pred):.4f} | F1: {f1_score(y_test, y_pred):.4f}")



Logistic Regression — Testing different max_iter
************************************************************
max_iter =    100 | Accuracy: 0.9474 | Recall: 0.9722 | F1: 0.9589
max_iter =    500 | Accuracy: 0.9649 | Recall: 0.9861 | F1: 0.9726
max_iter =   1000 | Accuracy: 0.9561 | Recall: 0.9722 | F1: 0.9655


/opt/anaconda3/envs/asl_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/envs/asl_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#log

max_iter =   5000 | Accuracy: 0.9649 | Recall: 0.9861 | F1: 0.9726
max_iter =  10000 | Accuracy: 0.9649 | Recall: 0.9861 | F1: 0.9726


In [31]:
print("SVM — Testing different C values")
print("*" * 60)
for c in [0.1, 0.5, 1, 5, 10]:
    model = SVC(C=c, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"C = {c:5} | Accuracy: {accuracy_score(y_test, y_pred):.4f} | Recall: {recall_score(y_test, y_pred):.4f} | F1: {f1_score(y_test, y_pred):.4f}")



SVM — Testing different C values
************************************************************
C =   0.1 | Accuracy: 0.9123 | Recall: 0.9722 | F1: 0.9333
C =   0.5 | Accuracy: 0.9211 | Recall: 0.9722 | F1: 0.9396
C =     1 | Accuracy: 0.9298 | Recall: 0.9722 | F1: 0.9459
C =     5 | Accuracy: 0.9211 | Recall: 0.9583 | F1: 0.9388
C =    10 | Accuracy: 0.9298 | Recall: 0.9722 | F1: 0.9459


In [32]:
print("KNN — Testing different n_neighbors values")
print("*" * 60)
for k in [3, 5, 7, 9, 11]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"K = {k:3} | Accuracy: {accuracy_score(y_test, y_pred):.4f} | Recall: {recall_score(y_test, y_pred):.4f} | F1: {f1_score(y_test, y_pred):.4f}")

KNN — Testing different n_neighbors values
************************************************************
K =   3 | Accuracy: 0.9298 | Recall: 0.9444 | F1: 0.9444
K =   5 | Accuracy: 0.9123 | Recall: 0.9167 | F1: 0.9296
K =   7 | Accuracy: 0.9298 | Recall: 0.9444 | F1: 0.9444
K =   9 | Accuracy: 0.9386 | Recall: 0.9583 | F1: 0.9517
K =  11 | Accuracy: 0.9386 | Recall: 0.9583 | F1: 0.9517
